# tensor-reshape-view — worked example 1: View and flatten on a contiguous tensor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-reshape-view`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`.view()` rearranges tensor dimensions without copying, but requires the tensor to be contiguous in memory. A freshly allocated tensor (from `torch.randn`, `torch.zeros`, etc.) is always contiguous. Calling `.view(-1)` or `.view(new_shape)` on such a tensor returns a view that shares the same storage — modifying one modifies the other.

## Worked solution

**Step 1 — Allocate a contiguous tensor.**
A tensor created with `torch.randn(B, C, H, W)` is stored in row-major (C-contiguous) order. `is_contiguous()` returns `True`.

**Step 2 — Flatten with view(-1).**
`.view(-1)` collapses all dimensions into one. For shape `(2, 3, 4)`, the result is shape `(24,)`. The `-1` is shorthand for "infer this dimension from the total number of elements".

**Step 3 — Reshape to a partial collapse.**
`.view(B, -1)` keeps the first dimension and collapses the rest. For `(2, 3, 4)`, this gives `(2, 12)`.

**Step 4 — Verify same storage.**
After `y = x.view(-1)`, modify `x[0, 0, 0] = 999` and print `y[0]`. They share storage, so `y[0]` will also show `999`.

In [ ]:
import torch as t

t.manual_seed(0)
x = t.randn(2, 3, 4)   # contiguous
print('x.shape:', x.shape)                     # (2, 3, 4)
print('x.is_contiguous():', x.is_contiguous()) # True

# Flatten completely
y_flat = x.view(-1)
print('view(-1) shape:', y_flat.shape)         # (24,)

# Partial collapse: keep first dim
y_2d = x.view(2, -1)
print('view(2, -1) shape:', y_2d.shape)        # (2, 12)

# Verify shared storage
x[0, 0, 0] = 999.0
print('y_flat[0] after mutation:', y_flat[0].item())   # 999.0 (shared)
print('y_2d[0, 0] after mutation:', y_2d[0, 0].item()) # 999.0 (shared)